In [3]:
import polars as pl
import pandas as pd
import numpy as np
import datetime as dt



In [4]:
## Read the CSV file
df = pl.read_csv("/home/abhishek/Desktop/BridgeLabz_Python_Mentor/PANDAS/sample_users.csv")

In [5]:
## Basic operations 

df.shape

(1000, 7)

In [6]:
## creating df using from json data :

df2 = pl.DataFrame(
    {
        "name" : ['abhi' , 'chirag' , 'niv' , 'sahil'],
        "age" : [dt.date(2000, 1,10) ,dt.date(2001, 1,10) , dt.date(2002, 1,10), dt.date(2000, 1,10)],
        "weight" : [90 , 40 , 70 , 30],
        "height" : [1.8 , 1.9 , 1.5 ,1.6]
    }
)

In [7]:
df2

name,age,weight,height
str,date,i64,f64
"""abhi""",2000-01-10,90,1.8
"""chirag""",2001-01-10,40,1.9
"""niv""",2002-01-10,70,1.5
"""sahil""",2000-01-10,30,1.6


In [16]:
df2.estimated_size()

98

In [17]:
## Print options

df.head(5)

UserID,Name,Age,Country,SignupDate,PurchaseAmount,IsPremium
str,str,i64,str,str,f64,bool
"""U1000""","""User_0""",40,"""UK""","""2023-01-01""",462.69,false
"""U1001""","""User_1""",23,"""UK""","""2023-01-02""",248.9,false
"""U1002""","""User_2""",28,"""India""","""2023-01-03""",888.86,true
"""U1003""","""User_3""",42,"""Australia""","""2023-01-04""",846.18,false
"""U1004""","""User_4""",26,"""Australia""","""2023-01-05""",106.44,true


In [18]:
type(df)

polars.dataframe.frame.DataFrame

*** Expressions and contexts ***

Expressions are one of the main strengths of Polars because they provide a modular and flexible way of expressing data transformations.



In [10]:
## Expression withh SELECT 

result = df2.select(
    pl.col("name"),
    pl.col("age").dt.year().alias("birth_year"),
    (pl.col("weight")/(pl.col("height")**2)).alias("bmi"),
)

In [11]:
print(result)

shape: (4, 3)
┌────────┬────────────┬───────────┐
│ name   ┆ birth_year ┆ bmi       │
│ ---    ┆ ---        ┆ ---       │
│ str    ┆ i32        ┆ f64       │
╞════════╪════════════╪═══════════╡
│ abhi   ┆ 2000       ┆ 27.777778 │
│ chirag ┆ 2001       ┆ 11.080332 │
│ niv    ┆ 2002       ┆ 31.111111 │
│ sahil  ┆ 2000       ┆ 11.71875  │
└────────┴────────────┴───────────┘


In Polars, ```expression expansion``` refers to how expressions (like pl.col("column_name")) can be automatically applied to multiple columns or dynamically expanded to generate a series of transformations — instead of writing them out manually for each column.

🧾 Example: Expression Expansion in Action
Let’s say you want to calculate the mean of several columns.

Instead of doing:

python
Copy
Edit
df.select([
    pl.col("math").mean(),
    pl.col("physics").mean(),
    pl.col("chemistry").mean()
])
You can do expression expansion like this:

python
Copy
Edit
df.select(
    pl.col(["math", "physics", "chemistry"]).mean()
)

```with_columns```

The context with_columns is very similar to the context select but with_columns adds columns to the dataframe instead of selecting them. Notice how the resulting dataframe contains the four columns of the original dataframe plus the two new columns introduced by the expressions inside with_columns:

In [12]:
result = df2.with_columns(
    bmi = pl.col("weight")/(pl.col("height") **3)

)

print(result)

shape: (4, 5)
┌────────┬────────────┬────────┬────────┬───────────┐
│ name   ┆ age        ┆ weight ┆ height ┆ bmi       │
│ ---    ┆ ---        ┆ ---    ┆ ---    ┆ ---       │
│ str    ┆ date       ┆ i64    ┆ f64    ┆ f64       │
╞════════╪════════════╪════════╪════════╪═══════════╡
│ abhi   ┆ 2000-01-10 ┆ 90     ┆ 1.8    ┆ 15.432099 │
│ chirag ┆ 2001-01-10 ┆ 40     ┆ 1.9    ┆ 5.831754  │
│ niv    ┆ 2002-01-10 ┆ 70     ┆ 1.5    ┆ 20.740741 │
│ sahil  ┆ 2000-01-10 ┆ 30     ┆ 1.6    ┆ 7.324219  │
└────────┴────────────┴────────┴────────┴───────────┘


```filter```
The context filter allows us to create a second dataframe with a subset of the rows of the original one:




In [13]:
result = df2.filter(
    pl.col("weight") > 40
)

print(result)

shape: (2, 4)
┌──────┬────────────┬────────┬────────┐
│ name ┆ age        ┆ weight ┆ height │
│ ---  ┆ ---        ┆ ---    ┆ ---    │
│ str  ┆ date       ┆ i64    ┆ f64    │
╞══════╪════════════╪════════╪════════╡
│ abhi ┆ 2000-01-10 ┆ 90     ┆ 1.8    │
│ niv  ┆ 2002-01-10 ┆ 70     ┆ 1.5    │
└──────┴────────────┴────────┴────────┘


```group_by```
The context group_by can be used to group together the rows of the dataframe that share the same value across one or more expressions. The example below counts how many people were born in each decade:

In [14]:
result = df.group_by(
    (pl.col("age").dt.year() // 10 * 10).alias("decade"),
    maintain_order=True,
).agg(
    pl.len().alias("sample_size"),
    pl.col("weight").mean().round(2).alias("avg_weight"),
    pl.col("height").max().alias("tallest"),
)
print(result)

ColumnNotFoundError: age

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["UserID", "Name", "Age", "Country", ...]; PROJECT */7 COLUMNS

In [19]:
df2.estimated_size()

98

In [29]:
df.describe()

statistic,UserID,Name,Age,Country,SignupDate,PurchaseAmount,IsPremium
str,str,str,f64,str,str,f64,f64
"""count""","""1000""","""1000""",1000.0,"""1000""","""1000""",1000.0,1000.0
"""null_count""","""0""","""0""",0.0,"""0""","""0""",0.0,0.0
"""mean""",null,null,37.737,null,null,542.82442,0.495
"""std""",null,null,12.299232,null,null,257.39165,null
"""min""","""U1000""","""User_0""",18.0,"""Australia""","""2023-01-01""",100.23,0.0
"""25%""",null,null,26.0,null,null,324.05,null
"""50%""",null,null,37.0,null,null,536.08,null
"""75%""",null,null,49.0,null,null,759.6,null
"""max""","""U1999""","""User_999""",59.0,"""USA""","""2025-09-26""",999.24,1.0


*** Handling missing values ***

In [38]:
## filling the missing value with 0

df = df.with_columns(
    pl.col("IsPremium").cast(pl.Float64).fill_null(0.0).alias("IsPremium")
)





In [33]:
df.describe()

statistic,UserID,Name,Age,Country,SignupDate,PurchaseAmount,IsPremium
str,str,str,f64,str,str,f64,f64
"""count""","""1000""","""1000""",1000.0,"""1000""","""1000""",1000.0,1000.0
"""null_count""","""0""","""0""",0.0,"""0""","""0""",0.0,0.0
"""mean""",null,null,37.737,null,null,542.82442,0.495
"""std""",null,null,12.299232,null,null,257.39165,null
"""min""","""U1000""","""User_0""",18.0,"""Australia""","""2023-01-01""",100.23,0.0
"""25%""",null,null,26.0,null,null,324.05,null
"""50%""",null,null,37.0,null,null,536.08,null
"""75%""",null,null,49.0,null,null,759.6,null
"""max""","""U1999""","""User_999""",59.0,"""USA""","""2025-09-26""",999.24,1.0


In [39]:
df = df.with_columns(
    pl.col("IsPremium")
    .cast(pl.Float64)  # ensure it's float
    .fill_null(0.0)    # fill nulls with 0.0
    .alias("IsPremium")
)


In [40]:
df.describe()

statistic,UserID,Name,Age,Country,SignupDate,PurchaseAmount,IsPremium
str,str,str,f64,str,str,f64,f64
"""count""","""1000""","""1000""",1000.0,"""1000""","""1000""",1000.0,1000.0
"""null_count""","""0""","""0""",0.0,"""0""","""0""",0.0,0.0
"""mean""",null,null,37.737,null,null,542.82442,0.495
"""std""",null,null,12.299232,null,null,257.39165,0.500225
"""min""","""U1000""","""User_0""",18.0,"""Australia""","""2023-01-01""",100.23,0.0
"""25%""",null,null,26.0,null,null,324.05,0.0
"""50%""",null,null,37.0,null,null,536.08,0.0
"""75%""",null,null,49.0,null,null,759.6,1.0
"""max""","""U1999""","""User_999""",59.0,"""USA""","""2025-09-26""",999.24,1.0


*** update column and intro to windows function ***


In [43]:
df

## 1. Update values on one or more columns 

new_Age_df = df.with_columns(
    (pl.col("Age")+10)

)
print(df)

shape: (1_000, 7)
┌────────┬──────────┬─────┬───────────┬────────────┬────────────────┬───────────┐
│ UserID ┆ Name     ┆ Age ┆ Country   ┆ SignupDate ┆ PurchaseAmount ┆ IsPremium │
│ ---    ┆ ---      ┆ --- ┆ ---       ┆ ---        ┆ ---            ┆ ---       │
│ str    ┆ str      ┆ i64 ┆ str       ┆ str        ┆ f64            ┆ f64       │
╞════════╪══════════╪═════╪═══════════╪════════════╪════════════════╪═══════════╡
│ U1000  ┆ User_0   ┆ 40  ┆ UK        ┆ 2023-01-01 ┆ 462.69         ┆ 0.0       │
│ U1001  ┆ User_1   ┆ 23  ┆ UK        ┆ 2023-01-02 ┆ 248.9          ┆ 0.0       │
│ U1002  ┆ User_2   ┆ 28  ┆ India     ┆ 2023-01-03 ┆ 888.86         ┆ 1.0       │
│ U1003  ┆ User_3   ┆ 42  ┆ Australia ┆ 2023-01-04 ┆ 846.18         ┆ 0.0       │
│ U1004  ┆ User_4   ┆ 26  ┆ Australia ┆ 2023-01-05 ┆ 106.44         ┆ 1.0       │
│ …      ┆ …        ┆ …   ┆ …         ┆ …          ┆ …              ┆ …         │
│ U1995  ┆ User_995 ┆ 40  ┆ USA       ┆ 2025-09-22 ┆ 915.18         ┆ 1.0       

*** Sorting and Ordering ***


In [21]:
print(df)

shape: (1_000, 7)
┌────────┬──────────┬─────┬───────────┬────────────┬────────────────┬───────────┐
│ UserID ┆ Name     ┆ Age ┆ Country   ┆ SignupDate ┆ PurchaseAmount ┆ IsPremium │
│ ---    ┆ ---      ┆ --- ┆ ---       ┆ ---        ┆ ---            ┆ ---       │
│ str    ┆ str      ┆ i64 ┆ str       ┆ str        ┆ f64            ┆ bool      │
╞════════╪══════════╪═════╪═══════════╪════════════╪════════════════╪═══════════╡
│ U1000  ┆ User_0   ┆ 40  ┆ UK        ┆ 2023-01-01 ┆ 462.69         ┆ false     │
│ U1001  ┆ User_1   ┆ 23  ┆ UK        ┆ 2023-01-02 ┆ 248.9          ┆ false     │
│ U1002  ┆ User_2   ┆ 28  ┆ India     ┆ 2023-01-03 ┆ 888.86         ┆ true      │
│ U1003  ┆ User_3   ┆ 42  ┆ Australia ┆ 2023-01-04 ┆ 846.18         ┆ false     │
│ U1004  ┆ User_4   ┆ 26  ┆ Australia ┆ 2023-01-05 ┆ 106.44         ┆ true      │
│ …      ┆ …        ┆ …   ┆ …         ┆ …          ┆ …              ┆ …         │
│ U1995  ┆ User_995 ┆ 40  ┆ USA       ┆ 2025-09-22 ┆ 915.18         ┆ true      

In [24]:
sorted_df  = df.sort("Age")
print(sorted_df)

shape: (1_000, 7)
┌────────┬──────────┬─────┬───────────┬────────────┬────────────────┬───────────┐
│ UserID ┆ Name     ┆ Age ┆ Country   ┆ SignupDate ┆ PurchaseAmount ┆ IsPremium │
│ ---    ┆ ---      ┆ --- ┆ ---       ┆ ---        ┆ ---            ┆ ---       │
│ str    ┆ str      ┆ i64 ┆ str       ┆ str        ┆ f64            ┆ bool      │
╞════════╪══════════╪═════╪═══════════╪════════════╪════════════════╪═══════════╡
│ U1006  ┆ User_6   ┆ 18  ┆ Australia ┆ 2023-01-07 ┆ 591.27         ┆ true      │
│ U1016  ┆ User_16  ┆ 18  ┆ India     ┆ 2023-01-17 ┆ 543.2          ┆ false     │
│ U1070  ┆ User_70  ┆ 18  ┆ India     ┆ 2023-03-12 ┆ 830.2          ┆ true      │
│ U1151  ┆ User_151 ┆ 18  ┆ Germany   ┆ 2023-06-01 ┆ 711.06         ┆ false     │
│ U1174  ┆ User_174 ┆ 18  ┆ Germany   ┆ 2023-06-24 ┆ 651.13         ┆ false     │
│ …      ┆ …        ┆ …   ┆ …         ┆ …          ┆ …              ┆ …         │
│ U1749  ┆ User_749 ┆ 59  ┆ Australia ┆ 2025-01-19 ┆ 424.43         ┆ false     

*** Handling missing values ***

In [25]:
print(df)

shape: (1_000, 7)
┌────────┬──────────┬─────┬───────────┬────────────┬────────────────┬───────────┐
│ UserID ┆ Name     ┆ Age ┆ Country   ┆ SignupDate ┆ PurchaseAmount ┆ IsPremium │
│ ---    ┆ ---      ┆ --- ┆ ---       ┆ ---        ┆ ---            ┆ ---       │
│ str    ┆ str      ┆ i64 ┆ str       ┆ str        ┆ f64            ┆ bool      │
╞════════╪══════════╪═════╪═══════════╪════════════╪════════════════╪═══════════╡
│ U1000  ┆ User_0   ┆ 40  ┆ UK        ┆ 2023-01-01 ┆ 462.69         ┆ false     │
│ U1001  ┆ User_1   ┆ 23  ┆ UK        ┆ 2023-01-02 ┆ 248.9          ┆ false     │
│ U1002  ┆ User_2   ┆ 28  ┆ India     ┆ 2023-01-03 ┆ 888.86         ┆ true      │
│ U1003  ┆ User_3   ┆ 42  ┆ Australia ┆ 2023-01-04 ┆ 846.18         ┆ false     │
│ U1004  ┆ User_4   ┆ 26  ┆ Australia ┆ 2023-01-05 ┆ 106.44         ┆ true      │
│ …      ┆ …        ┆ …   ┆ …         ┆ …          ┆ …              ┆ …         │
│ U1995  ┆ User_995 ┆ 40  ┆ USA       ┆ 2025-09-22 ┆ 915.18         ┆ true      

In [27]:
df

UserID,Name,Age,Country,SignupDate,PurchaseAmount,IsPremium
str,str,i64,str,str,f64,bool
"""U1000""","""User_0""",40,"""UK""","""2023-01-01""",462.69,false
"""U1001""","""User_1""",23,"""UK""","""2023-01-02""",248.9,false
"""U1002""","""User_2""",28,"""India""","""2023-01-03""",888.86,true
"""U1003""","""User_3""",42,"""Australia""","""2023-01-04""",846.18,false
"""U1004""","""User_4""",26,"""Australia""","""2023-01-05""",106.44,true
…,…,…,…,…,…,…
"""U1995""","""User_995""",40,"""USA""","""2025-09-22""",915.18,true
"""U1996""","""User_996""",39,"""USA""","""2025-09-23""",996.98,true
"""U1997""","""User_997""",23,"""USA""","""2025-09-24""",388.53,true


*** How to handle data larger then the RAM ***


Polars has streaming mode and lazy evaluation, which help handle large data without loading it all into memory at once — but this is not full out-of-core processing like Dask. Instead, it's more like streaming batches through memory-efficient execution.

In [ ]:
import polars as pl

lf = pl.read_csv("big_file.csv", has_header=True).lazy()

# Apply transformations lazily
result = (
    lf.filter(pl.col("value") > 100)
      .group_by("category")
      .agg(pl.col("value").mean())
      .collect(streaming=True)  # Important for large files
)


2. Read in Batches / Chunks

Use this approach when doing ETL or preprocessing on huge files.

In [ ]:

for batch in pl.read_csv("big_file.csv", chunk_size=100_000):
    df = batch.filter(pl.col("value") > 100)
    # Process or write to disk
